In [2]:
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

import numpy as np
import keras
import sentencepiece as spm

with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()



# Cell 1: Setup and imports
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'  # You can change to 'jax' or 'torch' if preferred

import numpy as np
import keras
import sentencepiece as spm

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.config.backend()}")

Keras version: 3.10.0
Keras backend: tensorflow


In [8]:
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Text length:", len(text))
print("\nFirst 200 characters:\n")
print(text[:200])

Text length: 5378702

First 200 characters:

﻿The Project Gutenberg eBook of The Complete Works of William Shakespeare
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with a


In [10]:
def clean_gutenberg_simple(text):
    # markers to indicate where book starts and ends (to skip the instructions)
    start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
    end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

    # Find position of start and end markers in the text
    start_idx = text.find(start_marker)
    end_idx = text.find(end_marker)

    # If the start marker is found, remove everything before it
    if start_idx != -1:
        text = text[start_idx + len(start_marker):]

    # If the end marker is found, remove everything after it
    if end_idx != -1:
        text = text[:end_idx]

    # Remove excessive empty lines
    while "\n\n\n" in text:
        text = text.replace("\n\n\n", "\n\n")

    return text.strip()


text = clean_gutenberg_simple(text)

print("Cleaned length:", len(text))
print("\nCleaned beginning of text:\n")
print(text[:200])

Cleaned length: 5357866

Cleaned beginning of text:

The Complete Works of William Shakespeare

by William Shakespeare

                    Contents

    THE SONNETS
    ALL’S WELL THAT ENDS WELL
    THE TRAGEDY OF ANTONY AND CLEOPATRA
    AS YOU LIKE I


In [11]:
# Save cleaned text to temporary file
temp_file = "shakespeare_temp.txt"
with open(temp_file, "w", encoding="utf-8") as f:
    f.write(text)

# Train SentencePiece tokenizer
vocab_size = 8000
model_prefix = "shakespeare_sp"

spm.SentencePieceTrainer.train(
    input=temp_file,
    model_prefix=model_prefix,
    vocab_size=vocab_size,
    character_coverage=1.0,
    model_type="bpe",
    user_defined_symbols=["<PAD>", "<UNK>"]
)

# Load tokenizer
sp = spm.SentencePieceProcessor()
sp.load(f"{model_prefix}.model")

# Test tokenization
test_text = "To be, or not to be, that is the question:"
tokens = sp.encode_as_pieces(test_text)

print("Tokenized example:", tokens)
print("Vocabulary size:", sp.get_piece_size())

Tokenized example: ['▁To', '▁be', ',', '▁or', '▁not', '▁to', '▁be', ',', '▁that', '▁is', '▁the', '▁question', ':']
Vocabulary size: 8000


In [12]:
# Convert full text into token IDs
pieces = sp.encode_as_ids(text)

print("Total tokens:", len(pieces))


# Sequence length (context size for the model)
seq_length = 96


# Split tokens into training and validation sets FIRST (important!)
split_idx = int(0.8 * len(pieces))

train_pieces = pieces[:split_idx]
val_pieces = pieces[split_idx:]


# Function to create input-target sequences
def create_sequences(token_ids, seq_length):
    sequences = []

    # Slide a window over the token list
    for i in range(len(token_ids) - seq_length):
        # Take seq_length + 1 tokens (input + next token)
        sequences.append(token_ids[i:i + seq_length + 1])

    sequences = np.array(sequences)

    # Inputs: all except last token
    inputs = sequences[:, :-1]

    # Targets: all except first token (shifted by 1)
    targets = sequences[:, 1:]

    return inputs, targets


# Create training and validation data
train_inputs, train_targets = create_sequences(train_pieces, seq_length)
val_inputs, val_targets = create_sequences(val_pieces, seq_length)


# Print shapes to verify
print("Train inputs shape:", train_inputs.shape)
print("Train targets shape:", train_targets.shape)

print("Validation inputs shape:", val_inputs.shape)
print("Validation targets shape:", val_targets.shape)

Total tokens: 1392234
Train inputs shape: (1113691, 96)
Train targets shape: (1113691, 96)
Validation inputs shape: (278351, 96)
Validation targets shape: (278351, 96)


In [15]:
def get_positional_encoding(max_len, d_model):
    positions = np.arange(max_len)[:, np.newaxis]
    angles = np.arange(d_model)[np.newaxis, :] / d_model
    angles = 1 / (10000**angles)

    pos_encoding = positions * angles
    pos_encoding[:, 0::2] = np.sin(pos_encoding[:, 0::2])
    pos_encoding[:, 1::2] = np.cos(pos_encoding[:, 1::2])

    return pos_encoding

In [16]:
embed_dim = 192      # size of token embeddings
num_heads = 4        # number of attention heads
ff_dim = 384         # feed-forward layer size
num_layers = 2       # number of transformer blocks

In [17]:
# Define model parameters
embed_dim = 192
num_heads = 4
ff_dim = 384
num_layers = 2

# Create the model
inputs = keras.Input(shape=(seq_length,))
embedding_layer = keras.layers.Embedding(sp.get_piece_size(), embed_dim)(inputs)

# Add positional encoding
pos_encoding = get_positional_encoding(seq_length, embed_dim)
x = embedding_layer + pos_encoding

# Transformer blocks
for _ in range(num_layers):
    # Multi-head attention with causal mask
    attention_output = keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_dim // num_heads,
        dropout=0.1
    )(x, x, use_causal_mask=True)

    # Add & Norm
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + attention_output)

    # Feed-forward network
    ffn = keras.Sequential([
        keras.layers.Dense(ff_dim, activation="relu"),
        keras.layers.Dense(embed_dim),
        keras.layers.Dropout(0.1)
    ])
    ffn_output = ffn(x)

    # Add & Norm
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + ffn_output)

# Final output layer
outputs = keras.layers.Dense(sp.get_piece_size())(x)

# Create model
model = keras.Model(inputs=inputs, outputs=outputs)

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 96)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 96, 192)   │  1,536,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 96, 192)   │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 96, 192)   │    148,224 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 96, 192)   │          0 │ add[0][0],        │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 96, 192)   │        384 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 96, 192)   │    148,032 │ layer_normalizat… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 96, 192)   │          0 │ layer_normalizat… │
│                     │                   │            │ sequential[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 96, 192)   │        384 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 96, 192)   │    148,224 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 96, 192)   │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 96, 192)   │        384 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 96, 192)   │    148,032 │ layer_normalizat… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 96, 192)   │          0 │ layer_normalizat… │
│                     │                   │            │ sequential_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 96, 192)   │        384 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 96, 8000)  │  1,544,000 │ layer_normalizat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,674,048 (14.02 MB)

 Trainable params: 3,674,048 (14.02 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
batch_size = 128
epochs = 100

history = model.fit(
    train_inputs, train_targets,
    validation_data=(val_inputs, val_targets),
    batch_size=batch_size,
    epochs=epochs,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1),
        keras.callbacks.ModelCheckpoint('kalevala_best_model.keras', save_best_only=True)
    ]
)

Epoch 1/100
  74/8701 ━━━━━━━━━━━━━━━━━━━━ 7:38:00 3s/step - accuracy: 0.0296 - loss: 8.7970


KeyboardInterrupt

